In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
import copy

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score
from sklearn.utils import class_weight

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.nn import Conv1d, LSTM, GRU
from torch.functional import F
from torch.utils.data import Dataset, TensorDataset, DataLoader
import pytorch_lightning
import torchmetrics
import torchsummary
from torch.nn import Sequential
from torch.nn.modules import Module
from torch.optim import Adam, Adamax, Adagrad, AdamW, Adafactor, Adadelta
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, ExponentialLR
#from torch.fft import fft, fftfreq, fftshift

from scipy.fft import fft

import skorch
from skorch import NeuralNetClassifier
from skorch.dataset import ValidSplit
from skorch.callbacks import EarlyStopping, Checkpoint, LRScheduler, EpochScoring, EpochTimer

from collections import defaultdict
import time

f:\Conda\envs\DeepLearning-Torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

cuda


# **Load Dataset**

In [3]:
X_windows = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X.npy")
y_categorical = np.load("F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y.npy")

In [4]:
X_windows = torch.FloatTensor(X_windows)
y_categorical = torch.FloatTensor(y_categorical)
y_encoded = y_integer = np.argmax(y_categorical, axis=1)

In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)
X_val, X_test, y_val, y_test = train_test_split(
    X_windows, y_categorical, test_size=0.5, random_state=42, stratify=y_categorical
)
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

Train: torch.Size([8089, 44, 50]), torch.Size([8089, 4])
Test: torch.Size([5056, 44, 50]), torch.Size([5056, 4])


In [6]:
# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [7]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [8]:
# Iterate through training data
for batch_idx, (eeg_data, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"  EEG data shape: {eeg_data.shape}")  # [batch_size, channels, time_points]
    print(f"  Labels shape: {labels.shape}")      # [batch_size]
    
    # Your model training code goes here
    # predictions = model(eeg_data)
    # loss = criterion(predictions, labels)
    
    if batch_idx == 2:  # Show first 3 batches
        break

# Check one batch from each loader
train_batch = next(iter(train_loader))
val_batch = next(iter(valid_loader))

print(f"\nTrain batch: {train_batch[0].shape}, {train_batch[1].shape}")
print(f"Validation batch: {val_batch[0].shape}, {val_batch[1].shape}")

Batch 0:
  EEG data shape: torch.Size([32, 44, 50])
  Labels shape: torch.Size([32, 4])
Batch 1:
  EEG data shape: torch.Size([32, 44, 50])
  Labels shape: torch.Size([32, 4])
Batch 2:
  EEG data shape: torch.Size([32, 44, 50])
  Labels shape: torch.Size([32, 4])

Train batch: torch.Size([32, 44, 50]), torch.Size([32, 4])
Validation batch: torch.Size([32, 44, 50]), torch.Size([32, 4])


In [9]:
X_train.shape

torch.Size([8089, 44, 50])

# **Xception Networks**

In [21]:
class XceptionBock(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()

        self.residual = nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=1, padding='same')

        self.main_path = Sequential(
            nn.ReLU(),
            nn.Conv1d(in_channels=in_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=in_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel),

            nn.ReLU(),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=8, padding='same', groups=out_channel),
            nn.Conv1d(in_channels=out_channel, out_channels=out_channel, kernel_size=1),
            nn.BatchNorm1d(out_channel)
        )

        #addition of residual and main tensor in forward pass
        
        self.pool = nn.MaxPool1d(2)
    def forward(self, x):
        res = self.residual(x)
        x = self.main_path(x)
        x = x + res
        x = self.pool(x)

        return x


class EEGXceptionNet_1(nn.Module):
    def __init__(self, input_shape, num_classes, name="Blahblahblah"):
        super().__init__()
        self.name=name
        channels, seq_length = input_shape

        self.input_block = Sequential(
            nn.Conv1d(in_channels=channels, out_channels=64, kernel_size=8, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.branch_convs = nn.ModuleList([
            Sequential(
                nn.Conv1d(in_channels=64, out_channels=32, kernel_size=ksize, padding='same', groups=32),
                nn.Conv1d(in_channels=32, out_channels=32, kernel_size=1),
                nn.BatchNorm1d(32),
                nn.ReLU(inplace=True)
            ) for ksize in [3,5,8,11]
        ]
        )
        #concatenation in forward pass

        self.post_concat = Sequential(
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=1, padding='same'),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.xception1 = XceptionBock(128, 128)
        self.xception2 = XceptionBock(128, 256)

        self.squeeze_and_excitation = Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 256//8),
            nn.ReLU(),
            nn.Linear(256//8, 256),
            nn.Sigmoid()
        )

        self.classifier = Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        #print("Input shape: ",x.shape)
        x = self.input_block(x)
        #print("Shape after input block: ", x.shape)
        branches = []
        for branch_conv in self.branch_convs:
            branches.append(branch_conv(x))
        #print("Branch shapes: ", branches[0].shape, branches[1].shape)
        x = torch.cat(branches, dim=1)
        #print("Shape after concatenation: ", x.shape)
        x = self.post_concat(x)
        #print("Shape after post_concat layer: ", x.shape)
        x = self.xception1(x)
        #print("Shape after xception layer 1: ", x.shape)
        x = self.xception2(x)
        #print("Shape after xception layer 2: ", x.shape)

        attn_weights = self.squeeze_and_excitation(x)
        #print("Shape after squeeze and excitation attention before temporal unsqueeze: ", x.shape)
        attn_weights = attn_weights.unsqueeze(2)
        #print("Shape after squeeze and excitation attention after temporal unsqueeze: ", x.shape)
        x = x*attn_weights
        #print("Shape after multiplication: ", x.shape)
        x = self.classifier(x)
        #print("Shape after classifier head: ", x.shape)

        return F.softmax(x)
        #return x

# **Training Loop**

In [11]:
#train_dict = {"Subject":[],"Model_name":[],"Accuracy":[],"Precision":[],"Recall":[],"F1_Score":[], "Kappa":[]}
#val_dict = {"Subject":[],"Model_name":[],"Accuracy":[],"Precision":[],"Recall":[],"F1_Score":[], "Kappa":[]}
train_dict = {"Epoch":[],"Accuracy":[],"Loss":[]}
val_dict = {"Epoch":[],"Accuracy":[],"Loss":[],"Precision":[],"Recall":[],"F1_Score":[], "Kappa":[]}

In [12]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

(torch.Size([8089, 44, 50]),
 torch.Size([8089, 4]),
 torch.Size([5056, 44, 50]),
 torch.Size([5056, 4]))

In [13]:
#declare model

model = EEGXceptionNet_1(input_shape=X_windows[0].shape, num_classes=4, name="Base_EEGXceptionNet")
#model = CNN_LSTM(input_shape=X[0].shape, name="Trial_CL")
model.to(device)
optimizer = AdamW(params=model.parameters(), lr=0.0005, weight_decay=0.0001)

def compute_class_weight(y_train):
    """Manually compute class weights."""
    y_train_classes = np.argmax(y_train, axis=1)
    num_classes = y_train.shape[1]
    
    # Count samples per class
    class_counts = np.bincount(y_train_classes, minlength=num_classes)
    
    # Compute weights: inversely proportional to class frequency
    total_samples = len(y_train_classes)
    class_weights = total_samples / (num_classes * class_counts)
    
    return dict(enumerate(class_weights))

y_dict = compute_class_weight(y_categorical)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor([y_dict[i] for i in range(0,4)]).to(device))
del y_dict

In [14]:
model.name

'Base_EEGXceptionNet'

In [15]:
model

EEGXceptionNet_1(
  (input_block): Sequential(
    (0): Conv1d(44, 64, kernel_size=(8,), stride=(1,), padding=same)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (branch_convs): ModuleList(
    (0): Sequential(
      (0): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=same, groups=32)
      (1): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
      (2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU(inplace=True)
    )
    (1): Sequential(
      (0): Conv1d(64, 32, kernel_size=(5,), stride=(1,), padding=same, groups=32)
      (1): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
      (2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU(inplace=True)
    )
    (2): Sequential(
      (0): Conv1d(64, 32, kernel_size=(8,), stride=(1,), padding=same, groups=32)
      (1): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
      (2): 

In [16]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model=model)

310116

In [17]:
X_windows.shape, X_windows[0].shape, y_categorical.shape, y_categorical[0].shape

(torch.Size([10112, 44, 50]),
 torch.Size([44, 50]),
 torch.Size([10112, 4]),
 torch.Size([4]))

# **Subject pooled**

In [27]:
def compute_class_weights(y_train):
    """Compute class weights for imbalanced datasets."""
    y_train_classes = np.argmax(y_train, axis=1)
    classes, counts = np.unique(y_train_classes, return_counts=True)
    total = len(y_train_classes)
    weights = total / (len(classes) * counts)
    weights = weights / weights.sum() * len(classes)
    return torch.tensor(weights, dtype=torch.float32)


class EarlyStopping:
    def __init__(self, patience=10, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.counter = 0
        self.best_loss = None
        self.best_model_state = None
        self.early_stop = False
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = copy.deepcopy(model.state_dict())
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        
        return self.early_stop
    
    def restore(self, model):
        if self.restore_best_weights and self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)


# ============ TRAINING LOOP ============

def train_model(model, train_loader, val_loader, num_classes=4, 
                learning_rate=0.001, weight_decay=1e-4, num_epochs=100,
                patience=20, min_lr=1e-6, device=None):
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model.to(device)
    print(f"Training on: {device}")
    
    # Loss with class weights
    if hasattr(train_loader.dataset, 'targets'):
        y_train = train_loader.dataset.targets
        if isinstance(y_train, torch.Tensor) and y_train.dim() > 1:
            y_train = torch.argmax(y_train, dim=1)
        class_weights = compute_class_weights(y_train.numpy() if isinstance(y_train, torch.Tensor) else y_train)
    else:
        class_weights = None
    
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    
    # Optimizer with weight decay (AdamW is better for weight decay)
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8, 
        min_lr=min_lr, verbose=True
    )
    
    # Early stopping
    early_stopping = EarlyStopping(patience=patience, min_delta=0.001)
    
    # Tracking
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_precision': [], 
        'val_recall': [], 'val_f1': [], 'val_kappa': [],
        'lr': []
    }
    
    best_val_accuracy = 0
    best_model_state = None
    
    print(f"\nStarting training for {num_epochs} epochs...")
    print(f"Learning rate: {learning_rate}, Weight decay: {weight_decay}")
    print(f"Early stopping patience: {patience}")
    print("-" * 50)
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            # Convert one-hot to class indices if needed
            if target.dim() > 1:
                target = torch.argmax(target, dim=1)
            
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, target)
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += target.size(0)
            train_correct += (predicted == target).sum().item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}')
        
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = 100 * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                
                if target.dim() > 1:
                    target_for_loss = torch.argmax(target, dim=1)
                else:
                    target_for_loss = target.clone()
                
                outputs = model(data)
                loss = criterion(outputs, target_for_loss)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                val_total += target_for_loss.size(0)
                val_correct += (predicted == target_for_loss).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(target_for_loss.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = 100 * val_correct / val_total
        
        # Calculate metrics
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        
        precision = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
        recall = recall_score(all_targets, all_preds, average='weighted', zero_division=0)
        f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(all_targets, all_preds)
        
        # Update learning rate
        scheduler.step(val_accuracy)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_accuracy)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_accuracy)
        history['val_precision'].append(precision)
        history['val_recall'].append(recall)
        history['val_f1'].append(f1)
        history['val_kappa'].append(kappa)
        history['lr'].append(current_lr)
        
        # Print epoch results
        print(f'\nEpoch [{epoch+1}/{num_epochs}]')
        print(f'  Train - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.2f}%')
        print(f'  Val   - Loss: {avg_val_loss:.4f}, Acc: {val_accuracy:.2f}%')
        print(f'  Val Metrics - Precision: {precision:.4f}, Recall: {recall:.4f}, '
              f'F1: {f1:.4f}, Kappa: {kappa:.4f}')
        print(f'  Learning Rate: {current_lr:.6f}')
        
        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_state,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_accuracy': val_accuracy,
                'val_loss': avg_val_loss,
                'history': history
            }, 'best_model.pth')
            print(f'  ✓ Saved best model (Accuracy: {val_accuracy:.2f}%)')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print(f'\nEarly stopping triggered after {epoch+1} epochs')
            early_stopping.restore(model)
            break
    
    print(f'\nTraining completed! Best validation accuracy: {best_val_accuracy:.2f}%')
    
    # Load best model
    checkpoint = torch.load('best_model.pth', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    return model, history


# ============ DATA PREPARATION ============

def prepare_data_loaders(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test, 
                        batch_size=32, num_workers=2):
    """
    Prepare PyTorch data loaders from numpy arrays.
    """
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train) if len(y_train.shape) > 1 else torch.LongTensor(y_train)
    
    X_val_t = torch.FloatTensor(X_val)
    y_val_t = torch.FloatTensor(y_val) if len(y_val.shape) > 1 else torch.LongTensor(y_val)
    
    # Transpose if needed: PyTorch expects (batch, channels, seq_len)
    if X_train_t.shape[1] != X_train_t.shape[-1]:  # If not channels first
        X_train_t = X_train_t.permute(0, 2, 1)
        X_val_t = X_val_t.permute(0, 2, 1)
        if X_test is not None:
            X_test_t = torch.FloatTensor(X_test).permute(0, 2, 1)
    
    # Create datasets
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                             shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                           shuffle=False, num_workers=num_workers, pin_memory=True)
    
    test_loader = None
    if X_test is not None and y_test is not None:
        y_test_t = torch.FloatTensor(y_test) if len(y_test.shape) > 1 else torch.LongTensor(y_test)
        test_dataset = TensorDataset(X_test_t, y_test_t)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, 
                                shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return train_loader, val_loader, test_loader


# ============ MAIN TRAINING PIPELINE ============

def train_eeg_xception_pipeline(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test,
                               input_shape=None, num_classes=None, model_type='improved',
                               use_class_weights=True):
    """
    Complete training pipeline matching TensorFlow implementation.
    """
    print("=" * 60)
    print(f"Training EEG Xception - PyTorch Implementation")
    print("=" * 60)
    
    # Determine input shape if not provided
    if input_shape is None:
        # PyTorch: (channels, seq_len)
        input_shape = (X_train.shape[2], X_train.shape[1])  # Assuming (batch, seq_len, channels)
    
    if num_classes is None:
        if len(y_train.shape) > 1:
            num_classes = y_train.shape[1]
        else:
            num_classes = len(np.unique(y_train))
    
    # Prepare data loaders
    print("\nPreparing data loaders...")
    train_loader, val_loader, test_loader = prepare_data_loaders(
        X_train, y_train, X_val, y_val, X_test, y_test,
        batch_size=32 if model_type == 'base' else 64
    )
    
    # Create model
    print(f"\nCreating {model_type} model...")
    model = EEGXceptionNet_1(input_shape, num_classes, name=f"EEG_Xception_{model_type}")
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Training configuration
    config = {
        'improved': {'lr': 0.0005, 'epochs': 500, 'patience': 25},
        'lightweight': {'lr': 0.001, 'epochs': 80, 'patience': 20},
        'base': {'lr': 0.001, 'epochs': 100, 'patience': 20}
    }.get(model_type, {'lr': 0.001, 'epochs': 100, 'patience': 20})
    
    # Train model
    print(f"\nStarting training...")
    trained_model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_classes=num_classes,
        learning_rate=config['lr'],
        weight_decay=1e-4,  # Match TensorFlow AdamW weight decay
        num_epochs=config['epochs'],
        patience=config['patience']
    )
    
    # Final evaluation on test set if available
    if test_loader is not None:
        print("\n" + "="*60)
        print("Final Evaluation on Test Set")
        print("="*60)
        
        trained_model.eval()
        test_correct = 0
        test_total = 0
        test_preds = []
        test_targets = []
        
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                
                if target.dim() > 1:
                    target = torch.argmax(target, dim=1)
                
                outputs = trained_model(data)
                _, predicted = torch.max(outputs.data, 1)
                
                test_total += target.size(0)
                test_correct += (predicted == target).sum().item()
                
                test_preds.extend(predicted.cpu().numpy())
                test_targets.extend(target.cpu().numpy())
        
        test_accuracy = 100 * test_correct / test_total
        print(f"Test Accuracy: {test_accuracy:.2f}%")
        
        # Calculate test metrics
        test_preds = np.array(test_preds)
        test_targets = np.array(test_targets)
        
        test_precision = precision_score(test_targets, test_preds, average='weighted', zero_division=0)
        test_recall = recall_score(test_targets, test_preds, average='weighted', zero_division=0)
        test_f1 = f1_score(test_targets, test_preds, average='weighted', zero_division=0)
        test_kappa = cohen_kappa_score(test_targets, test_preds)
        
        print(f"Test Precision: {test_precision:.4f}")
        print(f"Test Recall: {test_recall:.4f}")
        print(f"Test F1-Score: {test_f1:.4f}")
        print(f"Test Kappa: {test_kappa:.4f}")
    
    return trained_model, history


# ============ EXAMPLE USAGE ============

if __name__ == "__main__":
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Example: Assuming you have data loaded
    # X_train, y_train, X_val, y_val, X_test, y_test = load_your_data()
    
    # Train model
    model, history = train_eeg_xception_pipeline(
         X_train, y_train, X_test, y_test, X_test, y_test,
         model_type='improved'
     )
    
    print("\nCode ready for training!")

Using device: cuda
Training EEG Xception - PyTorch Implementation

Preparing data loaders...

Creating improved model...
Model parameters: 313,188
Trainable parameters: 313,188

Starting training...
Training on: cuda

Starting training for 500 epochs...
Learning rate: 0.0005, Weight decay: 0.0001
Early stopping patience: 25
--------------------------------------------------


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_3388\3814519184.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [1/500], Step [50/127], Loss: 1.3865
Epoch [1/500], Step [100/127], Loss: 1.3850

Epoch [1/500]
  Train - Loss: 1.3860, Acc: 26.23%
  Val   - Loss: 1.3853, Acc: 27.47%
  Val Metrics - Precision: 0.1413, Recall: 0.2747, F1: 0.1832, Kappa: 0.0241
  Learning Rate: 0.000500
  ✓ Saved best model (Accuracy: 27.47%)


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_3388\3814519184.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [2/500], Step [50/127], Loss: 1.3847
Epoch [2/500], Step [100/127], Loss: 1.3734

Epoch [2/500]
  Train - Loss: 1.3857, Acc: 27.02%
  Val   - Loss: 1.3840, Acc: 27.73%
  Val Metrics - Precision: 0.1422, Recall: 0.2773, F1: 0.1880, Kappa: 0.0282
  Learning Rate: 0.000500
  ✓ Saved best model (Accuracy: 27.73%)


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_3388\3814519184.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [3/500], Step [50/127], Loss: 1.3814
Epoch [3/500], Step [100/127], Loss: 1.3819

Epoch [3/500]
  Train - Loss: 1.3850, Acc: 27.33%
  Val   - Loss: 1.3830, Acc: 28.11%
  Val Metrics - Precision: 0.1460, Recall: 0.2811, F1: 0.1902, Kappa: 0.0337
  Learning Rate: 0.000500
  ✓ Saved best model (Accuracy: 28.11%)


KeyboardInterrupt: 

In [47]:
history = pd.DataFrame(history)

In [49]:
history.to_csv("F:/Project-Motor Imagery Classification/Checkpoints/EEGXceptionNet/log_all.csv")

In [48]:
history

,train_loss,train_acc,val_loss,val_acc,val_precision,val_recall,val_f1,val_kappa,lr
0,1.386972,25.713933,1.384490,27.848101,0.143838,0.278481,0.187959,0.030187,0.000500
1,1.386033,26.344418,1.386383,26.246044,0.145113,0.262460,0.150969,0.009861,0.000500
2,1.385112,26.974904,1.388645,27.037184,0.192610,0.270372,0.185314,0.018834,0.000500
3,1.385085,26.913092,1.383679,27.531646,0.254196,0.275316,0.213854,0.026453,0.000500
4,1.384516,28.248238,1.382042,28.382120,0.226456,0.283821,0.194307,0.037208,0.000500
...,...,...,...,...,...,...,...,...,...
171,0.779148,96.414884,1.009387,73.101266,0.732559,0.731013,0.731090,0.641087,0.000008
172,0.779312,96.464334,1.008595,73.101266,0.731801,0.731013,0.731017,0.641141,0.000008
173,0.778688,96.489059,1.008421,73.200158,0.733328,0.732002,0.732010,0.642421,0.000008
174,0.778840,96.439609,1.007889,73.279272,0.733738,0.732793,0.732771,0.643527,0.000008


# **Subject Dependent**

In [22]:
for sub in [1, 2, 3, 5, 6, 7, 8, 9]:
    X_windows = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/X{sub}.npy")
    y_categorical = np.load(f"F:/Project-Motor Imagery Classification/BCICIV_2a_gdf/Tensor/y{sub}.npy")

    X_windows = torch.FloatTensor(X_windows)
    y_categorical = torch.FloatTensor(y_categorical)
    y_encoded = y_integer = np.argmax(y_categorical, axis=1)
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_windows, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_windows, y_categorical, test_size=0.5, random_state=42, stratify=y_categorical
    )   
    def compute_class_weights(y_train):
        """Compute class weights for imbalanced datasets."""
        y_train_classes = np.argmax(y_train, axis=1)
        classes, counts = np.unique(y_train_classes, return_counts=True)
        total = len(y_train_classes)
        weights = total / (len(classes) * counts)
        weights = weights / weights.sum() * len(classes)
        return torch.tensor(weights, dtype=torch.float32)


    class EarlyStopping:
        def __init__(self, patience=10, min_delta=0, restore_best_weights=True):
            self.patience = patience
            self.min_delta = min_delta
            self.restore_best_weights = restore_best_weights
            self.counter = 0
            self.best_loss = None
            self.best_model_state = None
            self.early_stop = False
            
        def __call__(self, val_loss, model):
            if self.best_loss is None:
                self.best_loss = val_loss
                self.best_model_state = copy.deepcopy(model.state_dict())
            elif val_loss > self.best_loss - self.min_delta:
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_loss = val_loss
                self.best_model_state = copy.deepcopy(model.state_dict())
                self.counter = 0
            
            return self.early_stop
        
        def restore(self, model):
            if self.restore_best_weights and self.best_model_state is not None:
                model.load_state_dict(self.best_model_state)


    # ============ TRAINING LOOP ============

    def train_model(model, train_loader, val_loader, num_classes=4, 
                    learning_rate=0.001, weight_decay=1e-4, num_epochs=100,
                    patience=20, min_lr=1e-6, device=None):
        
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        model.to(device)
        print(f"Training on: {device}")
        
        # Loss with class weights
        if hasattr(train_loader.dataset, 'targets'):
            y_train = train_loader.dataset.targets
            if isinstance(y_train, torch.Tensor) and y_train.dim() > 1:
                y_train = torch.argmax(y_train, dim=1)
            class_weights = compute_class_weights(y_train.numpy() if isinstance(y_train, torch.Tensor) else y_train)
        else:
            class_weights = None
        
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
        
        # Optimizer with weight decay (AdamW is better for weight decay)
        optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        
        # Learning rate scheduler
        scheduler = ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=8, 
            min_lr=min_lr, verbose=True
        )
        
        # Early stopping
        early_stopping = EarlyStopping(patience=patience, min_delta=0.001)
        
        # Tracking
        history = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': [], 'val_precision': [], 
            'val_recall': [], 'val_f1': [], 'val_kappa': [],
            'lr': []
        }
        
        best_val_accuracy = 0
        best_model_state = None
        
        print(f"\nStarting training for {num_epochs} epochs...")
        print(f"Learning rate: {learning_rate}, Weight decay: {weight_decay}")
        print(f"Early stopping patience: {patience}")
        print("-" * 50)
        
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            train_loss = 0
            train_correct = 0
            train_total = 0
            
            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)
                
                # Convert one-hot to class indices if needed
                if target.dim() > 1:
                    target = torch.argmax(target, dim=1)
                
                optimizer.zero_grad()
                outputs = model(data)
                loss = criterion(outputs, target)
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                train_total += target.size(0)
                train_correct += (predicted == target).sum().item()
                
                if (batch_idx + 1) % 50 == 0:
                    print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], '
                        f'Loss: {loss.item():.4f}')
            
            avg_train_loss = train_loss / len(train_loader)
            train_accuracy = 100 * train_correct / train_total
            
            # Validation phase
            model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(device), target.to(device)
                    
                    if target.dim() > 1:
                        target_for_loss = torch.argmax(target, dim=1)
                    else:
                        target_for_loss = target.clone()
                    
                    outputs = model(data)
                    loss = criterion(outputs, target_for_loss)
                    val_loss += loss.item()
                    
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += target_for_loss.size(0)
                    val_correct += (predicted == target_for_loss).sum().item()
                    
                    all_preds.extend(predicted.cpu().numpy())
                    all_targets.extend(target_for_loss.cpu().numpy())
            
            avg_val_loss = val_loss / len(val_loader)
            val_accuracy = 100 * val_correct / val_total
            
            # Calculate metrics
            all_preds = np.array(all_preds)
            all_targets = np.array(all_targets)
            
            precision = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
            recall = recall_score(all_targets, all_preds, average='weighted', zero_division=0)
            f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
            kappa = cohen_kappa_score(all_targets, all_preds)
            
            # Update learning rate
            scheduler.step(val_accuracy)
            current_lr = optimizer.param_groups[0]['lr']
            
            # Update history
            history['train_loss'].append(avg_train_loss)
            history['train_acc'].append(train_accuracy)
            history['val_loss'].append(avg_val_loss)
            history['val_acc'].append(val_accuracy)
            history['val_precision'].append(precision)
            history['val_recall'].append(recall)
            history['val_f1'].append(f1)
            history['val_kappa'].append(kappa)
            history['lr'].append(current_lr)
            
            # Print epoch results
            print(f'\nEpoch [{epoch+1}/{num_epochs}]')
            print(f'  Train - Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.2f}%')
            print(f'  Val   - Loss: {avg_val_loss:.4f}, Acc: {val_accuracy:.2f}%')
            print(f'  Val Metrics - Precision: {precision:.4f}, Recall: {recall:.4f}, '
                f'F1: {f1:.4f}, Kappa: {kappa:.4f}')
            print(f'  Learning Rate: {current_lr:.6f}')
            
            # Save best model
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                best_model_state = copy.deepcopy(model.state_dict())
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': best_model_state,
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_accuracy': val_accuracy,
                    'val_loss': avg_val_loss,
                    'history': history
                }, f'EEGXception_{sub}.pth')
                print(f'  ✓ Saved best model (Accuracy: {val_accuracy:.2f}%)')
            
            # Early stopping check
            if early_stopping(avg_val_loss, model):
                print(f'\nEarly stopping triggered after {epoch+1} epochs')
                early_stopping.restore(model)
                break
        
        print(f'\nTraining completed! Best validation accuracy: {best_val_accuracy:.2f}%')
        
        # Load best model
        checkpoint = torch.load(f'EEGXception_{sub}.pth', map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        return model, history


    # ============ DATA PREPARATION ============

    def prepare_data_loaders(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test, 
                            batch_size=32, num_workers=2):
        """
        Prepare PyTorch data loaders from numpy arrays.
        """
        # Convert to PyTorch tensors
        X_train_t = torch.FloatTensor(X_train)
        y_train_t = torch.FloatTensor(y_train) if len(y_train.shape) > 1 else torch.LongTensor(y_train)
        
        X_val_t = torch.FloatTensor(X_val)
        y_val_t = torch.FloatTensor(y_val) if len(y_val.shape) > 1 else torch.LongTensor(y_val)
        
        # Transpose if needed: PyTorch expects (batch, channels, seq_len)
        if X_train_t.shape[1] != X_train_t.shape[-1]:  # If not channels first
            X_train_t = X_train_t.permute(0, 2, 1)
            X_val_t = X_val_t.permute(0, 2, 1)
            if X_test is not None:
                X_test_t = torch.FloatTensor(X_test).permute(0, 2, 1)
        
        # Create datasets
        train_dataset = TensorDataset(X_train_t, y_train_t)
        val_dataset = TensorDataset(X_val_t, y_val_t)
        
        # Create data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                                shuffle=True, num_workers=num_workers, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                            shuffle=False, num_workers=num_workers, pin_memory=True)
        
        test_loader = None
        if X_test is not None and y_test is not None:
            y_test_t = torch.FloatTensor(y_test) if len(y_test.shape) > 1 else torch.LongTensor(y_test)
            test_dataset = TensorDataset(X_test_t, y_test_t)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, 
                                    shuffle=False, num_workers=num_workers, pin_memory=True)
        
        return train_loader, val_loader, test_loader


    # ============ MAIN TRAINING PIPELINE ============

    def train_eeg_xception_pipeline(X_train, y_train, X_val, y_val, X_test=X_test, y_test=y_test,
                                input_shape=None, num_classes=None, model_type='improved',
                                use_class_weights=True):
        """
        Complete training pipeline matching TensorFlow implementation.
        """
        print("=" * 60)
        print(f"Training EEG Xception - PyTorch Implementation")
        print("=" * 60)
        
        # Determine input shape if not provided
        if input_shape is None:
            # PyTorch: (channels, seq_len)
            input_shape = (X_train.shape[2], X_train.shape[1])  # Assuming (batch, seq_len, channels)
        
        if num_classes is None:
            if len(y_train.shape) > 1:
                num_classes = y_train.shape[1]
            else:
                num_classes = len(np.unique(y_train))
        
        # Prepare data loaders
        print("\nPreparing data loaders...")
        train_loader, val_loader, test_loader = prepare_data_loaders(
            X_train, y_train, X_val, y_val, X_test, y_test,
            batch_size=32 if model_type == 'base' else 64
        )
        
        # Create model
        print(f"\nCreating {model_type} model...")
        model = EEGXceptionNet_1(input_shape, num_classes, name=f"EEG_Xception_{model_type}")
        
        print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
        
        # Training configuration
        config = {
            'improved': {'lr': 0.0005, 'epochs': 500, 'patience': 25},
            'lightweight': {'lr': 0.001, 'epochs': 80, 'patience': 20},
            'base': {'lr': 0.001, 'epochs': 100, 'patience': 20}
        }.get(model_type, {'lr': 0.001, 'epochs': 100, 'patience': 20})
        
        # Train model
        print(f"\nStarting training...")
        trained_model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            num_classes=num_classes,
            learning_rate=config['lr'],
            weight_decay=1e-4,  # Match TensorFlow AdamW weight decay
            num_epochs=config['epochs'],
            patience=config['patience']
        )
        
        # Final evaluation on test set if available
        if test_loader is not None:
            print("\n" + "="*60)
            print("Final Evaluation on Test Set")
            print("="*60)
            
            trained_model.eval()
            test_correct = 0
            test_total = 0
            test_preds = []
            test_targets = []
            
            with torch.no_grad():
                for data, target in test_loader:
                    data, target = data.to(device), target.to(device)
                    
                    if target.dim() > 1:
                        target = torch.argmax(target, dim=1)
                    
                    outputs = trained_model(data)
                    _, predicted = torch.max(outputs.data, 1)
                    
                    test_total += target.size(0)
                    test_correct += (predicted == target).sum().item()
                    
                    test_preds.extend(predicted.cpu().numpy())
                    test_targets.extend(target.cpu().numpy())
            
            test_accuracy = 100 * test_correct / test_total
            print(f"Test Accuracy: {test_accuracy:.2f}%")
            
            # Calculate test metrics
            test_history ={'test_acc': [], 'test_precision': [], 
            'test_recall': [], 'test_f1': [], 'test_kappa': []}
            test_preds = np.array(test_preds)
            test_targets = np.array(test_targets)
            
            test_precision = precision_score(test_targets, test_preds, average='weighted', zero_division=0)
            test_recall = recall_score(test_targets, test_preds, average='weighted', zero_division=0)
            test_f1 = f1_score(test_targets, test_preds, average='weighted', zero_division=0)
            test_kappa = cohen_kappa_score(test_targets, test_preds)

            test_history['test_acc'].append(test_accuracy)
            test_history['test_precision'].append(test_precision)
            test_history['test_recall'].append(test_recall)
            test_history['test_f1'].append(test_f1)
            test_history['test_kappa'].append(test_kappa)
            
            print(f"Test Precision: {test_precision:.4f}")
            print(f"Test Recall: {test_recall:.4f}")
            print(f"Test F1-Score: {test_f1:.4f}")
            print(f"Test Kappa: {test_kappa:.4f}")


        
        return trained_model, history, test_history


    # ============ EXAMPLE USAGE ============

    if __name__ == "__main__":
        # Set device
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
        
        # Example: Assuming you have data loaded
        # X_train, y_train, X_val, y_val, X_test, y_test = load_your_data()
        
        # Train model
        model, history, test_history = train_eeg_xception_pipeline(
            X_train, y_train, X_test, y_test, X_test, y_test,
            model_type='improved'
        )
        history = pd.DataFrame(history)
        history.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Albumentations/train_log_{sub}.csv")
        test_history = pd.DataFrame(test_history)
        test_history.to_csv(f"F:/Project-Motor Imagery Classification/Checkpoints/Albumentations/test_log_{sub}.csv")
        
        print("\nCode ready for training!")

Using device: cuda
Training EEG Xception - PyTorch Implementation

Preparing data loaders...

Creating improved model...
Model parameters: 329,572
Trainable parameters: 329,572

Starting training...
Training on: cuda

Starting training for 500 epochs...
Learning rate: 0.0005, Weight decay: 0.0001
Early stopping patience: 25
--------------------------------------------------


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_23136\1995639381.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [1/500], Step [50/58], Loss: 1.3918

Epoch [1/500]
  Train - Loss: 1.3873, Acc: 25.23%
  Val   - Loss: 1.3858, Acc: 26.87%
  Val Metrics - Precision: 0.2799, Recall: 0.2687, F1: 0.2562, Kappa: 0.0249
  Learning Rate: 0.000500
  ✓ Saved best model (Accuracy: 26.87%)


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_23136\1995639381.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [2/500], Step [50/58], Loss: 1.3769

Epoch [2/500]
  Train - Loss: 1.3852, Acc: 26.34%
  Val   - Loss: 1.3807, Acc: 29.08%
  Val Metrics - Precision: 0.2181, Recall: 0.2908, F1: 0.2133, Kappa: 0.0544
  Learning Rate: 0.000500
  ✓ Saved best model (Accuracy: 29.08%)


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_23136\1995639381.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


... (2,049 outputs left out)

C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_23136\1995639381.py:110: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.softmax(x)


Epoch [108/500], Step [50/58], Loss: 0.7903

Epoch [108/500]
  Train - Loss: 0.7802, Acc: 96.31%
  Val   - Loss: 1.0037, Acc: 73.70%
  Val Metrics - Precision: 0.7394, Recall: 0.7370, F1: 0.7371, Kappa: 0.6493
  Learning Rate: 0.000031

Early stopping triggered after 108 epochs

Training completed! Best validation accuracy: 73.78%

Final Evaluation on Test Set


C:\Users\Md Nazmus Saquib\AppData\Local\Temp\ipykernel_23136\621526526.py:228: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f'EEGXception_{sub}.pth'

Test Accuracy: 73.78%
Test Precision: 0.7425
Test Recall: 0.7378
Test F1-Score: 0.7380
Test Kappa: 0.6505

Code ready for training!
